# Mini RAG Chatbot

## Objective

Build a Retrieval-Augmented Generation (RAG) chatbot that can answer questions from a PDF document.

## RAG Pipeline

PDF → Text Extraction → Text Chunking → Embeddings → Vector Database → Semantic Search → Context Retrieval → LLM → Final Answer


In [2]:
pip install numpy

  Using cached numpy-2.5.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.5.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install pandas

  Using cached pandas-3.0.5-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
Using cached pandas-3.0.5-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (11.0 MB)
Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 526.3 kB/s eta 0:00:000:00:010:00:01:01
Note: you may need to restart the kernel to use updated packages.


In [13]:
pip install langchain

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 1.8 MB/s eta 0:00:00 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 352.7 kB/s eta 0:00:00MB/s eta 0:00:01
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.9/147.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 1.6 MB/s eta 0:00:002.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 1.7 MB/s eta 0:00:005.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 6.5 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 5.5 MB/s eta 0:00:005.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.8/161.8 kB 1.5 MB/s eta 0:00:008 MB/s eta 0:00:01
   ━━━━━━━━

In [6]:
# PDF reader
from pypdf import PdfReader

# Text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Gemini embeddings and chat model
from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

# FAISS vector database
from langchain_community.vectorstores import FAISS

# Operating system utilities
import os

print("All libraries imported successfully!")

All libraries imported successfully!


In [41]:
# Path to the new Machine Learning PDF

pdf_path = "sample_data/ch1_murphy.pdf"

print("PDF path:", pdf_path)

PDF path: sample_data/ch1_murphy.pdf


In [42]:
# Load the PDF
reader = PdfReader(pdf_path)

# Count the number of pages
num_pages = len(reader.pages)

print("PDF loaded successfully!")
print("Number of pages:", num_pages)

Ignoring wrong pointing object 150 0 (offset 0)
Ignoring wrong pointing object 154 0 (offset 0)
Ignoring wrong pointing object 196 0 (offset 0)
Ignoring wrong pointing object 226 0 (offset 0)
Ignoring wrong pointing object 256 0 (offset 0)
Ignoring wrong pointing object 260 0 (offset 0)
Ignoring wrong pointing object 277 0 (offset 0)


PDF loaded successfully!
Number of pages: 26


In [43]:
# Create an empty string to store the complete PDF text
text = ""

# Loop through every page in the PDF
for page in reader.pages:
    
    # Extract text from the current page
    page_text = page.extract_text()
    
    # Add the extracted text
    if page_text:
        text += page_text + "\n"

# Display the total number of characters
print("Total characters extracted:", len(text))

Total characters extracted: 58510


In [44]:
# Display the first 2000 characters
print(text[:2000])

’achine —earning
q ﬀrobabilistic ﬀerspective
–evin ﬀW ’urphy
ﬄhe ’yﬄ ﬀress
sambridgeU ’assachusetts
—ondonU ungland
1 Introduction
pmp Machine learningy what and why?
We are drowning in information and starving for knowledgeW — zohn “aisbittW
We are entering the era of big data W vor exampleU there are about a trillion web pages ak one
hour of video is uploaded to Y ouﬄ ube every secondU amounting to aZ years of content every
daybk the genomes of aZZZs of peopleU each of which has a length of 3.8 × 109 base pairsU have
been sequenced by various labsk Walmart handles more than a’ transactions per hour and has
databases containing more than bWe petabytes P 2.5 × 1015R of information Psukier bZaZRk and so
onW
ﬄhis deluge of data calls for automated methods of data analysisU which is what machine
learning providesW yn particularU we deﬁne machine learning as a set of methods that can
automatically detect patterns in dataU and then use the uncovered patterns to predict future
dataU or to pe

In [45]:
# Select the first 10 pages for our mini RAG project
# This keeps the number of embedding API requests small.

sample_text = ""

for page in reader.pages[:10]:
    page_text = page.extract_text()

    if page_text:
        sample_text += page_text + "\n"

print("Characters in selected pages:", len(sample_text))

Characters in selected pages: 22369


In [57]:
# Split the new PDF into chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200
)

chunks = text_splitter.split_text(text)

print("Number of chunks:", len(chunks))

Number of chunks: 46


In [58]:
# Create the Gemini embedding model

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

print("Gemini embedding model initialized successfully!")

Gemini embedding model initialized successfully!


In [59]:
# Create the FAISS vector database using the smaller chunk set

vector_store = FAISS.from_texts(
    small_chunks,
    embedding=embeddings
)

print("FAISS vector database created successfully!")

FAISS vector database created successfully!


In [49]:
# Define a test question
question = "What is machine learning?"

print("Question:", question)

Question: What is machine learning?


In [50]:
# Search the FAISS database for the 3 most relevant chunks

retrieved_docs = vector_store.similarity_search(
    question,
    k=3
)

# Display the retrieved chunks

for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Retrieved Chunk {i + 1} ---")
    print(doc.page_content)


--- Retrieved Chunk 1 ---
yn the simplest settingU each training input xi is a DVdimensional vector of numbersU repV
resentingU sayU the height and weight of a personW ﬄhese are called featuresU attributes or
covariatesW yn generalU howeverU xi could be a complex structured objectU such as an imageU a
sentenceU an email messageU a time seriesU a molecular shapeU a graphU etcW
ﬃimilarly the form of the output or response variable can in principle be anythingU but
most methods assume that yi is a categorical or nominal variable from some ﬁnite setU
yi ∈{ 1,...,C } Psuch as male or femaleRU or that yi is a realVvalued scalar Psuch as income
levelRW When yi is categoricalU the problem is known as classiﬁcation or pattern recognitionU
and when yi is realVvaluedU the problem is known as regressionW qnother variantU known as
ordinal regressionU occurs where label space Y has some natural orderingU such as grades q–vW
ﬄhe second main type of machine learning is the descriptive or unsupervised

In [51]:
# Initialize the Gemini language model
# This model will generate the final answer
# using the context retrieved from FAISS.

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

print("Gemini LLM initialized successfully!")

Gemini LLM initialized successfully!


In [52]:
# Test the Gemini LLM

response = llm.invoke(
    "Explain machine learning in one simple sentence."
)

print(response.content)

/home/nineleaps/mini-rag-chatbot/venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Machine learning is teaching computers to recognize patterns in data so they can learn to make decisions and predictions on their own, without being explicitly programmed.', 'extras': {'signature': 'EpMXCpAXARFNMg+fmAucPAv1nstNSLPSBA8Pa1Igz9X0VV0hiW5wa4W4PzK+HJFZgT6EEYR5OjGiEzI6GDLtK3CAm9tmmDbkXWcMKRS3cCv7cweQs4in9U+HtOBnH7kg/bRtXHAODhjmvwEQgPHI08948a2Fs8vHUT3lvPwmhM2sNIirvYMJ8vDD0+xV7JshM9SsuCL/KiMbk7KQhwbfXvLfB4eyykXmVlm2F2T1AlGACDI4dct5XBf6cBwaf3kAPJ4KC5Fszzg2aajVoyTQs+3NNZ0RGFGRLDMv1Oz/fi8cIP2YG/YL/Cbh6R2wiT97ok1KoI5xjZFvTUbxaKEeFQ8/ED2SPzfGJQHN3LDJugYUOkFzs9uHdASRFb+8ooaKN7ocS9SriJnAhgVN+2V7Px7jwueLqk1Ap3tEl0HcQMudUG37EnN8B2yWcQvzzZdOg66TVDnwWYRgvsMF04Ja9Y+AYEXzFLgbptrBMpAtPen33ce4qIN4MhrNFaMoFI5bL/wWbY1c/sm0AIqOlw7YK0vMzssY1KlgSfc/V1sNIoRy6A+GQE/6Loibk7+HDXXbRuiNbdgbKQw9hr14l0FJYi6IYc1AVSTCoxAJxuvRT3iljqghsXbci1FAfVopo86HiES3Tvvj2bpHV2rgmFm6K6siLO8wTf5CwoLIkvaBaAwu6jHOmMHa8O15/O8iEGG9akH8JUPr+5vTEHLhIvq9aJJwFuWrcxM8muZsuzTaH9p1N2fXe4DhwXVC+JZgpQO3QqxQYm

In [53]:
# User's question
question = "What is machine learning?"

print("Question:", question)

Question: What is machine learning?


In [54]:
# Retrieve the 3 most relevant chunks from the vector database

retrieved_docs = vector_store.similarity_search(
    question,
    k=3
)

# Display the retrieved chunks

for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Retrieved Chunk {i + 1} ---")
    print(doc.page_content)


--- Retrieved Chunk 1 ---
yn the simplest settingU each training input xi is a DVdimensional vector of numbersU repV
resentingU sayU the height and weight of a personW ﬄhese are called featuresU attributes or
covariatesW yn generalU howeverU xi could be a complex structured objectU such as an imageU a
sentenceU an email messageU a time seriesU a molecular shapeU a graphU etcW
ﬃimilarly the form of the output or response variable can in principle be anythingU but
most methods assume that yi is a categorical or nominal variable from some ﬁnite setU
yi ∈{ 1,...,C } Psuch as male or femaleRU or that yi is a realVvalued scalar Psuch as income
levelRW When yi is categoricalU the problem is known as classiﬁcation or pattern recognitionU
and when yi is realVvaluedU the problem is known as regressionW qnother variantU known as
ordinal regressionU occurs where label space Y has some natural orderingU such as grades q–vW
ﬄhe second main type of machine learning is the descriptive or unsupervised

In [55]:
# Combine the retrieved chunks into a single context

context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

print("Retrieved context:")
print(context)

Retrieved context:
yn the simplest settingU each training input xi is a DVdimensional vector of numbersU repV
resentingU sayU the height and weight of a personW ﬄhese are called featuresU attributes or
covariatesW yn generalU howeverU xi could be a complex structured objectU such as an imageU a
sentenceU an email messageU a time seriesU a molecular shapeU a graphU etcW
ﬃimilarly the form of the output or response variable can in principle be anythingU but
most methods assume that yi is a categorical or nominal variable from some ﬁnite setU
yi ∈{ 1,...,C } Psuch as male or femaleRU or that yi is a realVvalued scalar Psuch as income
levelRW When yi is categoricalU the problem is known as classiﬁcation or pattern recognitionU
and when yi is realVvaluedU the problem is known as regressionW qnother variantU known as
ordinal regressionU occurs where label space Y has some natural orderingU such as grades q–vW
ﬄhe second main type of machine learning is the descriptive or unsupervised learnin

In [56]:
# Create the RAG prompt

prompt = f"""
You are a helpful Machine Learning assistant.

Answer the user's question using ONLY the information
provided in the context below.

If the answer cannot be found in the context, say:
"I could not find the answer in the provided document."

Context:
{context}

Question:

{question}

Answer:
"""

print(prompt)


You are a helpful Machine Learning assistant.

Answer the user's question using ONLY the information
provided in the context below.

If the answer cannot be found in the context, say:
"I could not find the answer in the provided document."

Context:
yn the simplest settingU each training input xi is a DVdimensional vector of numbersU repV
resentingU sayU the height and weight of a personW ﬄhese are called featuresU attributes or
covariatesW yn generalU howeverU xi could be a complex structured objectU such as an imageU a
sentenceU an email messageU a time seriesU a molecular shapeU a graphU etcW
ﬃimilarly the form of the output or response variable can in principle be anythingU but
most methods assume that yi is a categorical or nominal variable from some ﬁnite setU
yi ∈{ 1,...,C } Psuch as male or femaleRU or that yi is a realVvalued scalar Psuch as income
levelRW When yi is categoricalU the problem is known as classiﬁcation or pattern recognitionU
and when yi is realVvaluedU the pro

In [37]:
# Send the RAG prompt to Gemini

response = llm.invoke(prompt)

# Display only the text answer
print("Answer:")
print(response.text)

/home/nineleaps/mini-rag-chatbot/venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Answer:
I could not find the answer in the provided document.


In [39]:
def ask_question(question):
    """
    Ask a question about the Machine Learning PDF.

    The function:
    1. Retrieves relevant chunks from FAISS
    2. Creates context from those chunks
    3. Builds a RAG prompt
    4. Sends the prompt to Gemini
    5. Returns the final answer
    """

    # Step 1: Retrieve relevant chunks
    retrieved_docs = vector_store.similarity_search(
        question,
        k=3
    )

    # Step 2: Combine retrieved chunks into context
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    # Step 3: Create the RAG prompt
    prompt = f"""
    You are a helpful Machine Learning assistant.

    Answer the user's question using ONLY the information
    provided in the context below.

    If the answer cannot be found in the context, say:
    "I could not find the answer in the provided document."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    # Step 4: Send prompt to Gemini
    response = llm.invoke(prompt)

    # Step 5: Return the clean text response
    return response.text

In [60]:
question = "Machine learning: what and why?"

answer = ask_question(question)

print("Question:", question)
print("\nAnswer:", answer)

/home/nineleaps/mini-rag-chatbot/venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question: Machine learning: what and why?

Answer: Based on the provided document:

* **Why it is needed:** We are entering the era of "big data" and are drowning in information (from sources such as web pages, YouTube video uploads, genomic sequencing, and retail transactions). This massive deluge of data requires automated methods of data analysis.
* **What it is:** Machine learning is defined as a set of methods that can automatically detect patterns in data, and then use those uncovered patterns to predict future data or to perform other kinds of decision-making under uncertainty (such as planning how to collect more data).


In [ ]:
# ============================================================
# MINI RAG CHATBOT - COMPLETE PIPELINE IN ONE CELL
# ============================================================

# -----------------------------
# 1. Import required libraries
# -----------------------------

import os
import tempfile

from IPython.display import display
import ipywidgets as widgets

from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

from langchain_community.vectorstores import FAISS


# ============================================================
# 2. Check Gemini API key
# ============================================================

if not os.getenv("GEMINI_API_KEY"):
    raise ValueError(
        "GEMINI_API_KEY was not found. "
        "Set your Gemini API key before running this cell."
    )

print("Gemini API key detected.")


# ============================================================
# 3. Create PDF upload widget
# ============================================================

upload_widget = widgets.FileUpload(
    accept="sample_data/ch1_murphy.pdf",
    multiple=False,
    description="Upload PDF"
)

display(upload_widget)

print("Please upload your PDF using the button above.")


# ============================================================
# 4. Wait for PDF upload
# ============================================================

# This function will run after the user uploads the PDF.

def process_pdf(change):

    if not upload_widget.value:
        return

    print("\nPDF uploaded. Starting RAG pipeline...\n")


    # --------------------------------------------------------
    # Get uploaded PDF
    # --------------------------------------------------------

    uploaded_file = list(upload_widget.value.values())[0]

    file_name = uploaded_file["name"]
    file_content = uploaded_file["content"]

    print("File:", file_name)


    # --------------------------------------------------------
    # Save uploaded PDF temporarily
    # --------------------------------------------------------

    temp_pdf = tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".pdf"
    )

    temp_pdf.write(file_content)
    temp_pdf.close()


    # --------------------------------------------------------
    # 5. Extract text from PDF
    # --------------------------------------------------------

    reader = PdfReader(temp_pdf.name)

    text = ""

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"


    print("Number of pages:", len(reader.pages))
    print("Characters extracted:", len(text))


    # --------------------------------------------------------
    # 6. Split PDF text into chunks
    # --------------------------------------------------------

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=200
    )

    chunks = text_splitter.split_text(text)

    print("Number of chunks:", len(chunks))


    # --------------------------------------------------------
    # 7. Create Gemini embeddings
    # --------------------------------------------------------

    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001"
    )

    print("Gemini embeddings created.")


    # --------------------------------------------------------
    # 8. Create FAISS vector database
    # --------------------------------------------------------

    vector_store = FAISS.from_texts(
        chunks,
        embedding=embeddings
    )

    print("FAISS vector database created.")


    # --------------------------------------------------------
    # 9. Create Gemini LLM
    # --------------------------------------------------------

    llm = ChatGoogleGenerativeAI(
        model="gemini-3.6-flash"
    )

    print("Gemini LLM ready.")


    # --------------------------------------------------------
    # 10. Ask the user for a question
    # --------------------------------------------------------

    question_box = widgets.Text(
        description="Question:",
        placeholder="Ask something about the PDF...",
        layout=widgets.Layout(width="80%")
    )

    ask_button = widgets.Button(
        description="Ask",
        button_style="primary"
    )

    output = widgets.Output()


    display(question_box)
    display(ask_button)
    display(output)


    # --------------------------------------------------------
    # 11. Question-answer function
    # --------------------------------------------------------

    def answer_question(button):

        question = question_box.value.strip()

        if not question:
            with output:
                print("Please enter a question.")

            return


        with output:

            output.clear_output()

            print("Question:", question)
            print("\nSearching the document...\n")


        # ----------------------------------------------------
        # Retrieve relevant chunks from FAISS
        # ----------------------------------------------------

        retrieved_docs = vector_store.similarity_search(
            question,
            k=3
        )


        # ----------------------------------------------------
        # Combine retrieved chunks into context
        # ----------------------------------------------------

        context = "\n\n".join(
            doc.page_content
            for doc in retrieved_docs
        )


        # ----------------------------------------------------
        # Create RAG prompt
        # ----------------------------------------------------

        prompt = f"""
You are a helpful assistant answering questions about a PDF document.

Use ONLY the information provided in the context below.

If the answer cannot be found in the context, say:

"I could not find the answer in the provided document."

Do not use outside knowledge.

---------------- CONTEXT ----------------

{context}

-------------- END CONTEXT --------------

Question:
{question}

Answer:
"""


        # ----------------------------------------------------
        # Generate answer using Gemini
        # ----------------------------------------------------

        response = llm.invoke(prompt)


        # ----------------------------------------------------
        # Display final answer
        # ----------------------------------------------------

        with output:

            output.clear_output()

            print("Question:")
            print(question)

            print("\nAnswer:")
            print(response.text)


    # --------------------------------------------------------
    # Connect button to function
    # --------------------------------------------------------

    ask_button.on_click(answer_question)


# ============================================================
# 12. Run process_pdf after upload
# ============================================================

upload_widget.observe(
    process_pdf,
    names="value"
)

In [62]:
pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 456.7 kB/s eta 0:00:00MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.3/217.3 kB 1.0 MB/s eta 0:00:003.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 4.7 MB/s eta 0:00:00 MB/s eta 0:00:01:01
Note: you may need to restart the kernel to use updated packages.
